Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
import trino
import pandas as pd

conn = trino.dbapi.connect(
    host="trino-tableau-prod.dp.navi-tech.in",
    port=443,
    user="databricks-user",
    catalog="awsdatacatalog",
    schema="default",
    http_scheme="https",
)


query = """

WITH base AS (
    SELECT
        canonical_ticket_id,
        overall_last_agent_name_for_sub_ticket AS agent_name,
        ticket_group AS product_type,
        ROW_NUMBER() OVER (
            PARTITION BY canonical_ticket_id
            ORDER BY ticket_created_time DESC
        ) AS rn
    FROM corp_analytics.cx_sub_ticket_data_model_tbl
    WHERE
        DATE(SUBSTRING(ticket_created_time, 1, 10)) >= DATE_TRUNC('month', current_date)
        AND DATE(SUBSTRING(ticket_created_time, 1, 10)) < current_date
        AND overall_last_agent_type_for_sub_ticket IN ('SUPPORT_AGENT_L1', 'SUPPORT_AGENT_L2')
        AND ticket_group IN (
            'BBPS', 'PERSONAL_LOAN', 'DIGITAL_GOLD', 'UPI',
            'CREDIT_LINE_UPI', 'AMC', 'GENERAL_INSURANCE', 'HOME_LOAN_SALES'
        )
        AND source_channel = 'CHAT'
        AND overall_last_agent_name_for_sub_ticket IS NOT NULL
)

SELECT
    agent_name       AS "Agent Name",
    product_type     AS "Product Type",
    COUNT(*)         AS "MTD Tickets Handled"
FROM base
WHERE rn = 1
GROUP BY 1, 2
ORDER BY agent_name, product_type

"""

df = pd.read_sql(query, conn)
display(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1sshtYrCvZS7lLufFwacEu8ZczfHmk4_U9QpUeFnPTb4/edit?gid=1963761199#gid=1963761199"
worksheet_name = "Agent Wise MTD Tickets Handled"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)